# 10 — Evaluation Labeling

## Goal

Manually label dot assignments for a sample of stories from each size bucket.
This creates a ground truth dataset for evaluating the dot detection pipeline.

## Approach

For each sampled story:
1. Inspect the articles (title, source, date)
2. Assign each article a dot number (0, 1, 2, ...) — articles about the same sub-event get the same number
3. Save the labels to JSON

Later: run the same articles through the pipeline and compare system output with these labels.

## Files
- `data/evaluation/sample_2_5.parquet` — sampled articles for bucket 2-5
- `data/evaluation/ground_truth_2_5.json` — manual dot labels for bucket 2-5

In [2]:
import json
import pickle
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

REPO_ROOT = Path("../").resolve()
EVAL_DIR = REPO_ROOT / "data" / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

# Load Data

In [3]:
df = pd.read_parquet(REPO_ROOT / "data/processed/articles_clean.parquet")
df["published_at_dt"] = pd.to_datetime(df["published_at_dt"], utc=True)

with open(REPO_ROOT / "data/processed/stories.pkl", "rb") as f:
    stories = pickle.load(f)

embs_all = np.load(REPO_ROOT / "data/processed/embeddings.npy")

print(f"Articles : {len(df):,}")
print(f"Stories  : {len(stories):,}")
print(f"Embeddings  : {len(embs_all):,}")

Articles : 46,143
Stories  : 18,473
Embeddings  : 30,462


In [4]:
def show_similarity(sid):
    arts = sorted(stories[sid], key=lambda i: df.iloc[i]["published_at_dt"])
    embs = embs_all[arts]
    sim = cosine_similarity(embs)

    print(f"Story {sid} — {len(arts)} articles")
    for local_idx, art_idx in enumerate(arts):
        row = df.iloc[art_idx]
        pub = row["published_at_dt"].strftime("%d %b %H:%M")
        fetched = pd.to_datetime(row["fetched_at"]).strftime("%d %b %H:%M")
        print(f"  [{local_idx}] {pub} [{fetched}] [{row['source']}] {row['title']}")

    print("\nCosine similarity matrix:")
    sim_df = pd.DataFrame(
        sim.round(2),
        index=[f"[{i}]" for i in range(len(arts))],
        columns=[f"[{i}]" for i in range(len(arts))],
    )
    print(sim_df.to_string())

In [5]:
BUCKETS = {
    "2-5": (2, 5),
    "5-10": (5, 10),
    "10-20": (10, 20),
    "20-50": (20, 50),
    "50+": (50, 70),
}


def sample_and_inspect(bucket: str, seed: int = RANDOM_SEED):
    lo, hi = BUCKETS[bucket]
    bucket_stories = {
        sid: arts for sid, arts in stories.items() if lo <= len(arts) <= hi
    }

    rng = np.random.default_rng(seed)
    shuffled = list(bucket_stories.items())
    rng.shuffle(shuffled)

    # 3 with different sizes, 2 random
    fixed, used_sizes = [], []
    for sid, arts in shuffled:
        if len(arts) not in used_sizes:
            fixed.append(sid)
            used_sizes.append(len(arts))
        if len(fixed) == 3:
            break

    remaining = [sid for sid, _ in shuffled if sid not in fixed]
    n_random = min(2, len(remaining))
    random_2 = rng.choice(remaining, size=n_random, replace=False).tolist() if n_random else []
    sampled = fixed + random_2

    for sid in sampled:
        arts = stories[sid]
        print(f"\n{'='*60}")
        print(f"Story {sid} — {len(arts)} articles  [bucket: {bucket}]")
        print(f"{'='*60}")
        for local_idx, art_idx in enumerate(
            sorted(arts, key=lambda i: df.iloc[i]["published_at_dt"])
        ):
            row = df.iloc[art_idx]
            pub = row["published_at_dt"].strftime("%d %b %H:%M")
            print(f"  [{local_idx}] {pub} [{row['source']}] {row['title']}")
            print(f"       {row['url']}")

    return sampled

In [6]:
def save_sample(sampled_sids, bucket: str):
    sample_indices = [art_idx for sid in sampled_sids for art_idx in stories[sid]]
    sample_df = df.iloc[sample_indices].copy()
    sample_df["story_id"] = [sid for sid in sampled_sids for _ in stories[sid]]
    out_path = EVAL_DIR / f"sample_{bucket.replace('-', '_')}.parquet"
    sample_df.to_parquet(out_path, index=True)
    print(f"Saved {len(sample_df)} articles → {out_path}")

In [7]:
def save_ground_truth(sampled_sids, manual_labels, bucket: str):
    ground_truth = {}
    for sid in sampled_sids:
        arts_sorted = sorted(stories[sid], key=lambda i: df.iloc[i]["published_at_dt"])
        ground_truth[str(sid)] = {
            "bucket": bucket,
            "articles": [df.iloc[i]["url"] for i in arts_sorted],
            "labels": manual_labels[sid],
        }
    out_path = EVAL_DIR / f"ground_truth_{bucket.replace('-', '_')}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(ground_truth, f, ensure_ascii=False, indent=2)
    print(f"Saved ground truth → {out_path}")

# Bucket 2-5

In [17]:
sampled_sids = sample_and_inspect("2-5")


Story 17716 — 2 articles  [bucket: 2-5]
  [0] 20 May 14:09 [24chasa] Кремъл: Политиците от балтийските държави поддържат маниакално антируски позиции
       https://www.24chasa.bg/mezhdunarodni/article/22887086
  [1] 20 May 18:09 [fakti] Говори Москва: Политиците от балтийските държави поддържат маниакално антируски позиции
       https://fakti.bg/world/1055698-govori-moskva-politicite-ot-baltiiskite-darjavi-poddarjat-maniakalno-antiruski-pozicii

Story 9785 — 5 articles  [bucket: 2-5]
  [0] 06 May 09:50 [bta] Джорджа Мелони ще се срещне в Рим с новоизбрания унгарски премиер Мадяр
       https://www.bta.bg/bg/news/world/1120268-dzhordzha-meloni-shte-se-sreshtne-v-rim-s-novoizbraniya-ungarski-premier-madyar
  [1] 06 May 09:59 [fakti] Мелони приема новия унгарски премиер Петер Мадяр в Рим
       https://fakti.bg/world/1052734-meloni-priema-novia-ungarski-premier-peter-madar-v-rim
  [2] 07 May 15:22 [nova] Петер Мадяр се срещна с Джорджа Мелони
       https://nova.bg/news/view/2026/05/07

In [12]:
show_similarity(sampled_sids[4])

Story 15864 — 2 articles
  [0] 15 May 09:43 [15 May 10:27] [actualno] Ясни са номерата в бюлетините за частичните избори на 14 юни
  [1] 15 May 09:58 [15 May 10:29] [vesti] ЦИК определи номерата в бюлетините за частичните избори на 14 юни

Cosine similarity matrix:
      [0]  [1]
  [0]  1.00  0.97
  [1]  0.97  1.00


In [13]:
manual_labels = {
    17716: [0, 0],
    9785: [0, 0, 1, 1, 1],
    2810: [0, 1, 2],
    265: [0, 0],
    15864: [0, 0],
}

In [ ]:
save_sample(sampled_sids, "2-5")

Saved 14 articles → /Users/ivanadonchevska/Projects/msc_thesis/data/evaluation/sample_2_5.parquet


In [ ]:
save_ground_truth(sampled_sids, manual_labels, "2-5")

Saved ground truth → /Users/ivanadonchevska/Projects/msc_thesis/data/evaluation/ground_truth_2_5.json


# Bucket 5-10

In [8]:
sampled_sids = sample_and_inspect("5-10")


Story 9249 — 5 articles  [bucket: 5-10]
  [0] 08 May 08:29 [24chasa] Вземете новите HONOR 600 Pro и HONOR 600 от A1 – в комплект с безжични слушалки и на специална цена
       https://www.24chasa.bg/biznes/article/22807226
  [1] 08 May 08:32 [actualno] Вземете новите HONOR 600 Pro и HONOR 600 от A1 – в комплект с безжични слушалки и на специална цена
       https://www.actualno.com/sciencelogy/vzemete-novite-honor-600-pro-i-honor-600-ot-a1-v-komplekt-s-bezjichni-slushalki-i-na-specialna-cena-news_2591422.html
  [2] 08 May 08:40 [fakti] Вземете новите HONOR 600 Pro и HONOR 600 от A1 – в комплект с безжични слушалки и на специална цена
       https://fakti.bg/technozone/1053138-vzemete-novite-honor-600-pro-i-honor-600-ot-a1-v-komplekt-s-bezjichni-slushalki-i-na-specialna-cena
  [3] 08 May 08:58 [vesti] Вземете новите HONOR 600 Pro и HONOR 600 от A1 – в комплект с безжични слушалки и на специална цена
       https://www.vesti.bg/lyubopitno/vzemete-novite-honor-600-pro-i-honor-600-ot-a1-v

In [10]:
manual_labels = {
    9249: [0, 0, 0, 0, 0],
    15502: [0, 0, 0, 0, 0, 0],
    9952: [0, 0, 0, 0, 0, 1, 1],
    10796: [0, 0, 0, 0, 0, 0, 0, 0],
    16390: [0, 0, 0, 0, 0, 0, 0, 1],
}

In [11]:
show_similarity(sampled_sids[1])

Story 15502 — 6 articles
  [0] 15 May 08:55 [15 May 10:28] [segabg] Средната заплата в София за пръв път мина 2000 евро
  [1] 15 May 09:27 [15 May 10:26] [fakti] НСИ: Средната заплата за март гони 1500 евро, а в София за първи път мина психологическата граница от 2000 евро
  [2] 15 May 09:57 [15 May 10:27] [24chasa] Средната заплата в България достига 1407 евро и нараства с 12,7% на годишна база през първото тримесечие
  [3] 15 May 11:14 [15 May 14:12] [actualno] Първи данни за заплатите в евро: Колко достигна средното възнаграждение през 2026 година?
  [4] 15 May 12:02 [15 May 14:13] [economic] Средната заплата в София надхвърли 2000 евро
  [5] 15 May 20:00 [15 May 21:04] [24chasa] Средна заплата в София вече над 2000 евро, но депутатите вземат два пъти повече (Обзор, графика)

Cosine similarity matrix:
      [0]   [1]   [2]   [3]   [4]   [5]
[0]  1.00  0.87  0.86  0.84  0.89  0.88
[1]  0.87  1.00  0.85  0.83  0.81  0.83
[2]  0.86  0.85  1.00  0.91  0.75  0.76
[3]  0.84  0.83  0.91  1

In [12]:
save_sample(sampled_sids, "5-10")

Saved 34 articles → /Users/ivanadonchevska/Projects/msc_thesis/data/evaluation/sample_5_10.parquet


In [13]:
save_ground_truth(sampled_sids, manual_labels, "5-10")

Saved ground truth → /Users/ivanadonchevska/Projects/msc_thesis/data/evaluation/ground_truth_5_10.json


# Bucket 10-20

In [30]:
sampled_sids = sample_and_inspect("10-20")


Story 7809 — 11 articles  [bucket: 10-20]
  [0] 05 May 11:28 [24chasa] България с 20 стрелци на европейското в Осиек
       https://www.24chasa.bg/sport/article/22789890
  [1] 06 May 06:32 [monitor] България ще бъде представена от 20 състезатели на Европейското първенство по спортна стрелба
       https://telegraph.bg/sport/oshte-sport/bylgariia-shte-byde-predstavena-ot-20-systezateli-na-evropejskoto-pyrvenstvo-po-sportna-strelba.-kakvo-mogat-da-napraviat-nashite-490627
  [2] 09 May 15:59 [bta] Добро представяне на българските стрелци в днешния ден на Европейското първенство по спортна стрелба в Осиек
       https://www.bta.bg/bg/news/sport/1122658-dobro-predstavyane-na-balgarskite-streltsi-v-dneshniya-den-na-evropeyskoto-parve
  [3] 10 May 09:41 [monitor] Как се представиха на българските стрелци в днешния ден на Европейското първенство по спортна стрелба в Осиек?
       https://telegraph.bg/sport/oshte-sport/kak-se-predstaviha-na-bylgarskite-strelci-v-dneshniia-den-na-evropejskoto-p

In [38]:
manual_labels = {
    7809: [0, 0, 1, 1, 2, 2, 2, 3, 3, 3, 4],
    6585: [0, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2],
    11154: [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 2],
    4738: [0, 0, 1, 1, 1, 1, 1, 1, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3],
    8716: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
}

In [37]:
show_similarity(sampled_sids[4])

Story 8716 — 10 articles
  [0] 08 May 16:40 [08 May 17:21] [nova] БНБ: 92.52% от левовете са изтеглени от обращение
  [1] 08 May 16:49 [08 May 17:21] [24chasa] БНБ: 92.52% от левовете са изтеглени от обращение
  [2] 08 May 17:12 [08 May 17:22] [standartnews] БНБ: Само 2,3 млрд. лева са в ръцете на хората
  [3] 08 May 17:16 [08 May 19:23] [vesti] БНБ: 92% от левовете вече са изтеглени от обращение
  [4] 08 May 17:16 [08 May 17:22] [bta] Над 92 на сто от левовете са изтеглени от обращение, съобщи БНБ
  [5] 08 May 18:00 [08 May 19:22] [fakti] 92.52% от левовете са изтеглени от обращение
  [6] 08 May 21:46 [08 May 22:57] [actualno] Каква част от левовете е изтеглена от обращение: Данни на БНБ
  [7] 09 May 06:13 [09 May 06:20] [segabg] 93% от левовете вече са прибрани в БНБ
  [8] 09 May 06:23 [09 May 09:22] [actualno] 2,3 милиарда лева все още са в българските джобове
  [9] 09 May 08:30 [09 May 09:23] [banker] Извън касите на БНБ са все още 2.3 млрд. лв. в банкноти и монети

Cosine similari

In [39]:
save_sample(sampled_sids, "10-20")

Saved 65 articles → /Users/ivanadonchevska/Projects/msc_thesis/data/evaluation/sample_10_20.parquet


In [40]:
save_ground_truth(sampled_sids, manual_labels, "10-20")

Saved ground truth → /Users/ivanadonchevska/Projects/msc_thesis/data/evaluation/ground_truth_10_20.json


# Bucket 20-50

In [51]:
sampled_sids = sample_and_inspect("20-50")


Story 5614 — 20 articles  [bucket: 20-50]
  [0] 29 Apr 21:35 [actualno] Тръмп: Обмисляме намаляване на броя на американските войници в Германия
       https://www.actualno.com/america/trymp-obmisljame-namaljavane-na-broja-na-amerikanskite-vojnici-v-germanija-news_2588236.html
  [1] 30 Apr 01:44 [fakti] Вашингтон обмисля намаляване на броя на американските въоръжени сили в Германия
       https://fakti.bg/world/1051433-vashington-obmisla-namalavane-na-broa-na-amerikanskite-vaorajeni-sili-v-germania
  [2] 30 Apr 03:15 [nova] След спор с Мерц: Тръмп обмисля намаляване на американските войници в Германия
       https://nova.bg/news/view/2026/04/30/535600/%D1%81%D0%BB%D0%B5%D0%B4-%D1%81%D0%BF%D0%BE%D1%80-%D1%81-%D0%BC%D0%B5%D1%80%D1%86-%D1%82%D1%80%D1%8A%D0%BC%D0%BF-%D0%BE%D0%B1%D0%BC%D0%B8%D1%81%D0%BB%D1%8F-%D0%BD%D0%B0%D0%BC%D0%B0%D0%BB%D1%8F%D0%B2%D0%B0%D0%BD%D0%B5-%D0%BD%D0%B0-%D0%B0%D0%BC%D0%B5%D1%80%D0%B8%D0%BA%D0%B0%D0%BD%D1%81%D0%BA%D0%B8%D1%82%D0%B5-%D0%B2%D0%BE%D0%B9%D0%BD%D0%B8%

In [47]:
manual_labels = {
    5614: [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 3, 4, 5, 5, 6, 7, 7, 8, 9],
    344: [0, 1, 1, 2, 3, 4, 4, 5, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 14, 15, 15, 16, 17, 17, 18, 19, 20],
    14047: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 2, 1, 3, 4, 0, 4, 4, 0, 0, 0, 0, 5, 6, 6, 7, 7, 6, 8, 9, 9, 10, 10 ],
    8049: [0, 1, 2, 3, 4, 5, 6, 7, 8, 8, 9, 9, 10, 11, 12, 13, 13, 14, 15, 15, 16, 17],
    6673: [0, 0, 0, 0, 1, 2, 3, 4, 4, 4, 5, 5, 5, 5, 6, 6, 6, 7, 8, 8, 8, 8, 8, 8, 9, 9, 9 ],
}

In [50]:
show_similarity(sampled_sids[4])

Story 6673 — 27 articles
  [0] 25 Apr 20:08 [25 Apr 20:41] [24chasa] Мъри Стоилов и Филип Кръстев запазиха жива мечтата за Европа
  [1] 26 Apr 07:27 [26 Apr 09:04] [fakti] Гьозтепе на Мъри Стоилов с първа победа от четири мача
  [2] 26 Apr 09:21 [26 Apr 10:49] [24chasa] Мъри Стоилов: Ако сме единни, може да стигнем до Европа
  [3] 26 Apr 11:02 [26 Apr 13:12] [actualno] Европа е все по-близо: Гьозтепе на Мъри Стоилов прегази Анталияспор със светкавичен рекорд в турската Суперлига
  [4] 02 May 21:26 [02 May 22:45] [24chasa] Тимът на Мъри изпусна да бие 10 от "Трабзон", шанс за Европа още има
  [5] 03 May 11:36 [03 May 13:16] [24chasa] Мъри Стоилов: "Гьозтепе" не се плаши от никого и преследва мечтата си
  [6] 04 May 12:55 [04 May 14:07] [24chasa] Мъри Стоилов: За "Гьозтепе" идва най-важният мач през пролетта
  [7] 07 May 10:09 [07 May 10:28] [bta] Станимир Стоилов е пред своя мач номер 100 начело на турския Гьозтепе
  [8] 07 May 10:17 [07 May 10:27] [24chasa] Мъри Стоилов записва мач №10

In [48]:
save_sample(sampled_sids, "20-50")

Saved 130 articles → /Users/ivanadonchevska/Projects/msc_thesis/data/evaluation/sample_20_50.parquet


In [49]:
save_ground_truth(sampled_sids, manual_labels, "20-50")

Saved ground truth → /Users/ivanadonchevska/Projects/msc_thesis/data/evaluation/ground_truth_20_50.json


# Bucket 50+

In [8]:
sampled_sids = sample_and_inspect("50+")


Story 11769 — 51 articles  [bucket: 50+]
  [0] 06 May 17:45 [24chasa] Две българки участват в ключови местни избори в Англия
       https://www.24chasa.bg/mezhdunarodni/article/22797100
  [1] 07 May 02:22 [bta] Местни избори в Англия и парламентарни избори в Уелс и Шотландия
       https://www.bta.bg/bg/news/world/1120533-mestni-izbori-v-angliya-i-parlamentarni-izbori-v-uels-i-shotlandiya
  [2] 07 May 06:34 [vesti] Местни избори във Великобритания: Първи голям тест за властта на Киър Стармър
       https://www.vesti.bg/sviat/mestni-izbori-vyv-velikobritaniia-pyrvi-goliam-test-za-vlastta-na-kiyr-starmyr-6258893
  [3] 08 May 07:23 [bta] Първите резултати от местните избори във Великобритания сочат поражения за лейбъристите на премиера Стармър и победи за реформаторите на Фараж
       https://www.bta.bg/bg/news/world/1121509-parvite-rezultati-ot-mestnite-izbori-vav-velikobritaniya-sochat-porazheniya-za-l
  [4] 08 May 07:33 [24chasa] След първите резултати - партията на британския премиер

In [16]:
manual_labels = {
    11769: [0, 0, 0, 1, 1, 2, 1, 2, 1, 2, 2, 1, 3, 3, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 9, 9, 9, 9, 9, 10],
    10122: [0, 0, 0, 0, 1, 1, 1, 2, 2, 3, 3, 2, 4, 3, 4, 5, 4, 5, 5, 5, 5, 5, 6, 6, 7, 8, 7, 7, 7, 8, 8, 8, 7, 8, 9, 9, 9, 9, 8, 9, 9, 11, 10, 10, 9, 10, 10, 11, 12, 12, 12, 12, 11, 13, 14, 13, 14, 15, 15, 16, 16, 16, 15, 16, 15],
    7842: [0, 0, 0, 1, 1, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7, 10, 8, 8, 8, 9, 9, 9, 9, 11, 11, 11, 11, 11, 11, 10, 10, 11, 10, 10, 12, 12, 12, 12, 12, 12, 12, 13, 13, 12, 14, 14, 15, 15, 15, 15, 15],
    13438: [0, 0, 0, 0, 0, 0, 1, 1, 2, 2, 2, 1, 3, 3, 4, 4, 4, 4, 3, 3, 5, 5, 5, 6, 6, 6, 6, 6, 7, 8, 8, 8, 7, 8, 9, 9, 10, 10, 10, 12, 11, 11, 11, 12, 12, 12, 12, 12, 13, 13, 13, 14, 14, 14, 12, 15, 15, 15, 15, 15, 15, 16, 16, 16, 16],
    3077: [0, 1, 2, 3, 3, 1, 2, 4, 4, 2, 2, 4, 5, 5, 5, 6, 6, 7, 7, 8, 8, 9, 8, 10, 9, 10, 10, 11, 11, 11, 11, 12, 11, 13, 13, 14, 14, 14, 15, 15, 15, 16, 15, 15, 17, 17, 18, 18, 18, 19, 20, 20, 20, 20, 20, 21, 22, 22, 22, 20, 23, 23, 23, 24, 24, 24, 24, 24, 12, 25, 25, 25, 25, 25, 26, 26, 26, 12, 26],
}

In [12]:
show_similarity(sampled_sids[3])

Story 13438 — 65 articles
  [0] 01 May 07:52 [01 May 09:48] [24chasa] Пътят Смолян – Пампорово е пропаднал, има и заледени участъци в планината
  [1] 01 May 09:44 [01 May 09:48] [fakti] Слягане на пътното платно е установено по пътя Смолян – Пампорово, има заледени участъци в планината
  [2] 01 May 11:15 [01 May 13:24] [24chasa] Пътят Смолян - Пампорово пропадна, затвориха го напълно
  [3] 01 May 12:12 [01 May 17:00] [nova] Борове, движещи се към пропаст: Свлачище затвори пътя Пампорово-Смолян
  [4] 01 May 13:22 [01 May 14:58] [24chasa] Дървета и стълбове се сриват около пропадналия участък на пътя край Пампорово, има опасност да рухне и сграда (Снимки, видео)
  [5] 01 May 15:56 [01 May 17:00] [actualno] Пропадане на платното заради свлачище: Пътят Смолян - Пампорово е затворен
  [6] 01 May 17:56 [01 May 19:16] [24chasa] Кметът на Смолян обяви бедствено положение заради свлачището край Пампорово
  [7] 01 May 18:14 [01 May 19:16] [fakti] Кметът на Смолян Николай Мелемов обяви бедствено 

In [13]:
# the (50, 70) range only contains 4 stories — draw a 5th from the larger ones
candidates = sorted(
    sid for sid, arts in stories.items() if len(arts) > 70 and len(arts) < 80 and sid not in sampled_sids
)
rng = np.random.default_rng(RANDOM_SEED)
extra_sid = int(rng.choice(candidates))
sampled_sids.append(extra_sid)

arts = stories[extra_sid]
print(f"\n{'='*60}")
print(f"Story {extra_sid} — {len(arts)} articles  [bucket: 50+]")
print(f"{'='*60}")
for local_idx, art_idx in enumerate(
    sorted(arts, key=lambda i: df.iloc[i]["published_at_dt"])
):
    row = df.iloc[art_idx]
    pub = row["published_at_dt"].strftime("%d %b %H:%M")
    print(f"  [{local_idx}] {pub} [{row['source']}] {row['title']}")
    print(f"       {row['url']}")


Story 3077 — 79 articles  [bucket: 50+]
  [0] 21 Apr 17:08 [24chasa] Над 2400 души са загинали в Ливан при конфликта между "Хизбула" и Израел
       https://www.24chasa.bg/mezhdunarodni/article/22709599
  [1] 22 Apr 03:29 [actualno] Друго нарушено примирие: „Хизбула“ изстреля ракети срещу Израел
       https://www.actualno.com/asia/drugo-narusheno-primirie-hizbula-izstrelja-raketi-sreshtu-izrael-news_2584664.html
  [2] 22 Apr 20:16 [nova] Израелски удари убиха четирима души в Ливан
       https://nova.bg/news/view/2026/04/22/534793/%D0%B8%D0%B7%D1%80%D0%B0%D0%B5%D0%BB%D1%81%D0%BA%D0%B8-%D1%83%D0%B4%D0%B0%D1%80%D0%B8-%D1%83%D0%B1%D0%B8%D1%85%D0%B0-%D1%87%D0%B5%D1%82%D0%B8%D1%80%D0%B8%D0%BC%D0%B0-%D0%B4%D1%83%D1%88%D0%B8-%D0%B2-%D0%BB%D0%B8%D0%B2%D0%B0%D0%BD/
  [3] 23 Apr 04:35 [24chasa] Ливанска журналистка е убита при израелски удар
       https://www.24chasa.bg/mezhdunarodni/article/22717706
  [4] 23 Apr 07:38 [fakti] Ливан обвини Израел във военни престъпления след удар, при който з

In [15]:
show_similarity(sampled_sids[4])

Story 3077 — 79 articles
  [0] 21 Apr 17:08 [22 Apr 06:02] [24chasa] Над 2400 души са загинали в Ливан при конфликта между "Хизбула" и Израел
  [1] 22 Apr 03:29 [22 Apr 06:03] [actualno] Друго нарушено примирие: „Хизбула“ изстреля ракети срещу Израел
  [2] 22 Apr 20:16 [22 Apr 20:58] [nova] Израелски удари убиха четирима души в Ливан
  [3] 23 Apr 04:35 [23 Apr 06:08] [24chasa] Ливанска журналистка е убита при израелски удар
  [4] 23 Apr 07:38 [23 Apr 09:47] [fakti] Ливан обвини Израел във военни престъпления след удар, при който загина журналистка
  [5] 24 Apr 04:05 [24 Apr 06:10] [24chasa] „Хизбула“ е изстреляла от Ливан ракети по северната част на Израел
  [6] 24 Apr 13:20 [24 Apr 13:41] [fakti] Двама загинали при израелски удар в Южен Ливан въпреки примирието
  [7] 25 Apr 07:56 [25 Apr 08:56] [fakti] Израел атакува заредени ракетни установки в Ливан
  [8] 25 Apr 08:58 [25 Apr 10:48] [24chasa] Израелските сили атакуваха ракетни установки на "Хизбула"
  [9] 25 Apr 12:43 [25 Apr 13:10]

In [17]:
save_sample(sampled_sids, "50+")

Saved 330 articles → /Users/ivanadonchevska/Projects/msc_thesis/data/evaluation/sample_50+.parquet


In [18]:
save_ground_truth(sampled_sids, manual_labels, "50+")

Saved ground truth → /Users/ivanadonchevska/Projects/msc_thesis/data/evaluation/ground_truth_50+.json
